import simvoice


In [1]:
from sympy import *

## Lossless Tube Junction


The total pressure $P$ and flow $U$ of a tube section can be expressed by the foward and backward partial pressures ($F$ and $B$):
$$\begin{align}
P &= F+B\\
U &= Z^{-1}(F-B)
\end{align}$$
When two tubes of different diameters are connected, $P$ and $U$ must remain constant in both tubes: i.e., $P_1=P_2$ and $U_1=U_2$

In [2]:
Z1, Z2, F1, B2, D = symbols("Z_1,Z_2,F_1,B_2,D")

M = Matrix([[1, -1], [1 / Z2, 1 / Z1]])
M2 = Matrix([[1, -1], [1 / Z1, 1 / Z2]])

F = M**-1 * M2 * Matrix([[F1], [B2]])  # - Matrix([[F1], [B2]])
F2 = collect(F[0] - F1, (F1, B2))
B1 = collect(F[1] - B2, (F1, B2))
display(F2)
display(B1)

B_2*(Z_1/(Z_1 + Z_2) - Z_2/(Z_1 + Z_2)) + F_1*(2*Z_2/(Z_1 + Z_2) - 1)

B_2*(2*Z_1/(Z_1 + Z_2) - 1) + F_1*(-Z_1/(Z_1 + Z_2) + Z_2/(Z_1 + Z_2))

$$
\begin{aligned}
\begin{bmatrix}F_2\\B_1\end{bmatrix}
&= \begin{bmatrix}F_1\\B_2\end{bmatrix}
+ \frac{1}{Z_1+Z_2}
\begin{bmatrix}
-Z_1+Z_2 & Z_1-Z_2\end{bmatrix}
\begin{bmatrix}F_1\\B_2\end{bmatrix}
\end{aligned}
$$
where
$$
Z_k = \frac{\rho c}{A_k}
$$
with the crosssectional area $A_k$ of the $k-th$ tube section, air density $\rho$,
and speed of sound $c$.

# Story-95 Model: Tube Junction with Yielding Wall with Simplified Viscous and Heat Conduction Losses


The lossless model is expanded to include various per-unit length loss factors: yielding wall, viscous/laminar loss, heat coduction loss, and kinetic pressure drop.

<div style="background-color: aliceblue;">
  <img src="story95_model.svg">
</div>

- $R_w$ - wall resistance (simulates the viscosity of the tissue/energy dissipation)
  $$
  R_w = \frac{B}{2l\sqrt{\pi A}}
  $$
- $L_w$ - wall inertance (mass-like qualities of the flesh)
  $$
  L_w = \frac{M}{2l\sqrt{\pi A}}
  $$
- $C_w$ - wall compliance (simulates pressure storage capability)
  $$
  C_w = \frac{B}{2l\sqrt\pi A}
  $$
- viscous + laminar resistance $R_v$
- viscous resistance
  $$
  R_{vsc} = \frac{S}{A^2} \left( \frac{\omega \mu \rho}{2} \right)^\frac{1}{2} l
  $$
- laminar resistance
  $$
  R_{lam} = \frac{8\pi\mu}{A^2}l
  $$
- viscous inertance
  $$
  L_{vsc} = \frac{S}{A^2} \left( \frac{\mu \rho}{2 \omega} \right)^\frac{1}{2} l
  $$
- heat conduction gain, $\gamma$
- kinetic pressure drop, $P_e$

with $S = 2\sqrt{A\pi}$ is the circumference of a tube section and $\omega$ is
fixed as $6280$ rad/s. Let $R_v \triangleq R_{vsc} + R_{lam}$.
Here, $M=1.5$ g/cm^2, $B=1060$ dyne s/cm^3, $K = 33000$ dyne/cm^3


To convert to a wave-reflection model, we again establish the pressure and flow conservations:
$$\begin{align}
P_1 - P_v + P_e &= \gamma^{-1} P_2 = P,\\
U_1 &= U_w + \gamma^{-1} U_2 = U_w + U.
\end{align}$$

The transfer function of the yielding wall circuit is found by in the Laplace domain to define the relationship between $P \triangleq P_1+P_e$ and $U_w$:
$$\begin{aligned}
H_w(s) \triangleq \frac{U_w(s)}{P(s)} &= \left[R_w + sL_w + \frac{1}{sC_w}\right]^{-1}\\
                                      &= \frac{C_w s}{L_w C_w s^2 + RC_w s + 1}
\end{aligned}\tag{3}$$

To simplify the model, the change in the viscous loss $P_v$ is assumed decoupled from the tube junction conditions and yielding wall, thus the viscous loss circuit is simply replaced with a pressure source for this part of the model. The pressure will be adjusted independently.

$H_w$ has internal states ($\mathbf{s}_w$). Let $\mathbf{A}_w$, $\mathbf{b}_w$, $\mathbf{c}_w$, and $d_w$ constitute a state-space representation of $H_w$ such that
$$\begin{align}
\dot{\mathbf{s}}_w &= \mathbf{A}_w \mathbf{s}_w + \mathbf{b}_w P \tag{4}\\
             U_w &= \mathbf{c}_w \mathbf{s}_w + d_w P \tag{5}
\end{align}
$$


Substituting (5) into (2) yields
$$\begin{equation}U_1 = \mathbf{c}_w \mathbf{s}_w + d_w P + U \tag{6}\end{equation}$$

Substituting $H_w(s)$, the partial pressures $F_1$, $B_1$, $F_2$, $B_2$ and tube impedance $Z_1$ and $Z_2$ into (1) and (2):
$$\begin{aligned}
F_1 + B_1 - P_{ve} &= F + B\\
Z_1^{-1}(F_1 - B_1) &= \mathbf{c}_w \mathbf{s}_w + d_w (F+B) + Z_2^{-1}(F-B),
\end{aligned}$$
where $P_{ve} \triangleq P_v - P_e$.

Gather the output $F$ and $B_1$ to the left hand side:
$$\begin{aligned}
                 F -          B_1 &=          F_1 -                 B - P_{ve}\\
(Z_2^{-1} + d_w) F + Z_1^{-1} B_1 &= Z_1^{-1} F_1 + (Z_2^{-1} -d_w) B - \mathbf{c}_w\mathbf{s}_w\\
\end{aligned}$$

Convert to the matrix-vector format:
$$
\begin{bmatrix}
             1 & -1\\
Z_2^{-1} + d_w & Z_1^{-1}
\end{bmatrix}
\begin{bmatrix}F\\B_1\end{bmatrix}
=
\begin{bmatrix}
             1 & -1\\
Z_1^{-1} & Z_2^{-1}-d_w
\end{bmatrix}
\begin{bmatrix}F_1\\B\end{bmatrix}
+ 
\begin{bmatrix}
-1 & 0 \\
0 & -1
\end{bmatrix}
\begin{bmatrix}
P_{ve}\\
\mathbf{c}_w \mathbf{s}_w
\end{bmatrix}
$$

Solve this equation for $\begin{bmatrix}F&B_1\end{bmatrix}^T$:

In [3]:
dw, Y1, Y2, D = symbols("d_w,Y_1,Y_2,D")
M = Matrix([[1, -1], [Y2 + dw, Y1]])
dexpr = Y1 + Y2 + dw
display(simplify(M**-1 * Matrix([[1, -1], [Y1, Y2 - dw]])) * dexpr)
display(simplify(M**-1 * Matrix([[-1, 0], [0, -1]])) * dexpr)

Matrix([
[          2*Y_1, -Y_1 + Y_2 - d_w],
[Y_1 - Y_2 - d_w,            2*Y_2]])

Matrix([
[     -Y_1, -1],
[Y_2 + d_w, -1]])

$$
\begin{bmatrix}F\\B_1\end{bmatrix}=
\frac{1}{D}
\begin{bmatrix}
2 Z_1^{-1}                  & - Z_1^{-1} + Z_2^{-1} - d_w\\
  Z_1^{-1} - Z_2^{-1} - d_w & 2 Z_2^{-1}
\end{bmatrix}
\begin{bmatrix}F\\B_1\end{bmatrix}
+
\frac{1}{D}
\begin{bmatrix}
-Z_1^{-1}\\
 Z_2^{-2} + d_w
\end{bmatrix}
P_{ve}
-
\frac{\mathbf{c}_w}{D} \mathbf{s}_w
$$
where
$$
D \triangleq Z_1^{-1} + Z_2^{-1} + d_w
$$

Additional algebraic manipulation to mimic the lossless form yields
$$
\begin{bmatrix}F\\B_1\end{bmatrix}
=
\begin{bmatrix}F\\B_1\end{bmatrix}
+
\frac{Z_1^{-1} - Z_2^{-1}}{D}
(F-B_1)
- \frac{d_w}{D}
\begin{bmatrix}F\\B_1\end{bmatrix}
+
\frac{1}{D}
\begin{bmatrix}
-Z_1^{-1}\\
Z_2^{-2} + d_w
\end{bmatrix}
P_{ve} - \frac{\mathbf{c}_w}{D} \mathbf{s}_w \tag{7}
$$

Finally, this form can be written in terms of the cross-sectional areas $A_1 = c \rho Z_1^{-1}$ and $A_2 = c\rho Z_2^{-1}$ and use the Story95 parameterization:
$$
\begin{bmatrix}F\\B_1\end{bmatrix}
=
\begin{bmatrix}F\\B_1\end{bmatrix}
+
\Psi
+
\frac{1}{D}
\begin{bmatrix} -A_1\\A_2 + d_w \rho c \end{bmatrix}
P_{ve} - \frac{\mathbf{c}_w \rho c}{D} \mathbf{s}_w
\tag{8}$$
where
$$\begin{aligned}
\Psi &= r_1 F_1 + r_2 B\\
r_1 &= \frac{A_1 - A_2 -d_w \rho c}{D}\\
r_2 &= \frac{A_2 - A_1 -d_w \rho c}{D}\\
D &= A_1 + A_2 + d_w \rho c \\
\end{aligned}$$
The yielding wall internal states are updated by (4). This formulation generalizes Story95 (2.70) and (2.71) with the equivalences in the discretized version (to be shown): $\alpha = d_w$, $\beta = \mathbf{c}_w\mathbf{s}_w/D$, and (4) is equivalent to (2.45).

Note that this formulation works in the both continuous- and discrete-time domains. The discrete-time state space equations are:
$$\begin{align}
\mathbf{s}_{w,n+1} &= \mathbf{A}_w \mathbf{s}_{w,n} + \mathbf{b}_w p_n \tag{9}\\
u_{w,n} &=\mathbf{c}_w \mathbf{s}_{w,n} + d_w p_n \tag{10}\\
\end{align}

The viscous loss system of Tube 1 has the transfer function

$$
H_v(s) \triangleq \frac{P_v(s)}{U_1(s)} = R_v + sL_v\tag{11}
$$

### Discretization via the bilinear transform


The model is converted from to the discrete-time domain via the bilinear transformation.

Bilinear transformation converts continuous-time TF to discrete-time by substituting
$$
s = \frac{2}{T}\frac{1-z^{-1}}{1+z^{-1}}
$$


Story95 converted the each continuous-time block in Fig. 2-5 to discrete-time:

$$\begin{aligned}
P_l &= P - P_r - P_c\\
\frac{U_w}{P_l} &= \frac{1+z^{-1}}{2\tilde{L}_w(1-z^{-1})} = \frac{\alpha_l (1+z^{-1})}{1-z^{-1}} \triangleq H_l(z)\\
\frac{P_r}{U_w} &= R_w = \alpha_r\\
\frac{P_c}{U_w} &= \frac{1+z^{-1}}{2\tilde{C}_w(1-z^{-1})} = \frac{\alpha_c (1+z^{-1})}{1-z^{-1}} \triangleq H_c(z)\\
\end{aligned}$$

With these 1st order models, the signal flow graph of connected system is established and selected $U_w$ and $P_c$ as the states (named $\beta_l$ and $\beta_c$):

<img src="story95_wall_sfg.png">

Here, we have two states: $\beta_{l,n-1}$ and $\beta_{c,n-1}$. The update equation for the output $u_{w,n}$ is then written as
$$\begin{aligned}
u_{w,n} &= \alpha_l p_{l,n} + \beta_{l,n-1} \\
&= \alpha_l (p_n - p_{c,n} - p_{r,n}) + \beta_{l,n-1}\\
&= \alpha_l p_n - \alpha_l(\alpha_c u_{w,n} + \beta_{c,n-1}) - \alpha_l p_{r,n} + \beta_{l,n-1}\\
u_{w,n} + \alpha_l \alpha_c u_{w,n} + \alpha_l \alpha_r u_{w,n}&= \alpha_l p_n - \alpha_l \beta_{c,n-1} + \beta_{l,n-1}\\
(1 + \alpha_l \alpha_c + \alpha_l \alpha_r) u_{w,n}&= \alpha_l p_n - \alpha_l \beta_{c,n-1} + \beta_{l,n-1}\\
u_{w,n}&= \frac{\alpha_l}{1 + \alpha_l \alpha_c + \alpha_l \alpha_r} p_n - \frac{\alpha_l \beta_{c,n-1} - \beta_{l,n-1}}{1 + \alpha_l \alpha_c + \alpha_l \alpha_r}\\
&= \alpha p_n + \beta_n
\end{aligned} \tag{2.46}$$

From the signal flow graph, the state update equestions are found as:
$$\begin{align}
\beta_{c,n} &= \alpha_c u_{w,n} + \beta_{c,n-1} \tag{2.48}\\
\beta_{l,n} &= \alpha_l (p_n - \alpha_r u_{w,n} - \beta_{c,n} - \beta_{c,n-1}) = \alpha_l p_n - \alpha_l (\alpha_c + \alpha_r) u_{w,n} - \alpha_l \beta_{c,n-1} + \beta_{l,n-1} \tag{2.47}\\
\begin{bmatrix}
\beta_{l,n}\\
\beta_{c,n}
\end{bmatrix}
&= 
\begin{bmatrix}
1 & -\alpha_l\\
0 & 1
\end{bmatrix}
\begin{bmatrix}
\beta_{l,n-1}\\
\beta_{c,n-1}
\end{bmatrix}
+ 
\begin{bmatrix}
\alpha_l & -\alpha_l(\alpha_c + \alpha_r)\\
0 & \alpha_c
\end{bmatrix}
\begin{bmatrix}
p_n\\
u_{w,n}
\end{bmatrix}\notag
\end{align}$$

Hence, the discrete-time statespace coefficients are
$$\begin{aligned}
\mathbf{A}_w &= \begin{bmatrix}
1 & -\alpha_l\\
0 & 1
\end{bmatrix}+\begin{bmatrix}
-\alpha_l(\alpha_c+\alpha_r)\\\alpha_c\end{bmatrix}\mathbf{c}_w\\
\mathbf{b}_w &= \begin{bmatrix}
\alpha_l\\
0
\end{bmatrix}
+\begin{bmatrix}
-\alpha_l(\alpha_c+\alpha_r)\\\alpha_c\end{bmatrix}d_w\\
\mathbf{c}_w &= \begin{bmatrix}
\frac{1}{1+\alpha_l(\alpha_c+\alpha_r)} & -\frac{-\alpha_l}{1+\alpha_l(\alpha_c+\alpha_r)}
\end{bmatrix}\\
d_w &= \frac{\alpha_l}{1+\alpha_l(\alpha_c+\alpha_r)}
\end{aligned}$$

Here we confirm that $d_w=\alpha$ of this model. Note that (8) does not require to perform the detailed circuit analysis and can rely on a canonical tranfer function to states space converter.

The discretization of the series viscous loss is approximatd with a backward difference
such that $s = (1-z^{-1})/T$. Substituting $s$ of (10) yields
$$
H_v(s) = R_v + L_v / T (1-z^{-1}).
$$
The update equation is then
$$
p_{v,n} = R_v u_n + L_v/T (u_n - u_{n-1}),\tag{12}
$$
which matches (2.73) if $L_v$ is constant.

### Implementation

Initialization
* Calculate the DT statespace matrices of the yielding wall: $\mathbf{A}_w$, $\mathbf{b}_w$, $\mathbf{c}_w$, $d_w$
* Calculate coefficients for (8): $r_1$, $r_2$, $d_w \rho c$, $\mathbf{c}_w \rho c/D$

Updating the $k$th tube during the $n$th loop:

0. Assumptions:

    - Tube $(k-1)$ has been updated with output $f_{k,n}$ as well as derived $u_{k,n}$.
    - Tube $(k+1)$'s last backward output $b_{k,n}$ is has been calculated.
1. Calculate the viscous loss $p_{v,n}$ with (12)
2. Calculate the kinetic pressure drop $p_{e,n}$ with (2.91) (if the narrowest else 0)
3. Calculate $\mathbf{c}_w \mathbf{s}_{w,n}^{(k)} \rho c/D$
4. Calculate $b_n^{(k)} = \gamma b_{k,n}$
5. Calculate $f_n^{(k)}$ and $b_{k-1,n}$ by (8)
6. Calculate $f_{k,n} = f_n^{(k)}$
7. Calculate $\mathbf{s}_{w,n+1}^{(k)}$ by (9)


# T-model

Story95 wave-reflection model limits the interaction between the tube-junction 
effect and the tube characteristics only to the yielding wall effect while the 
other effects, namely the viscous loss and heat conduction loss, were not coupled 
with the junction effect. A natural extension of the model is to tie the latter
effects to the junciton and yielding wall effects.

<div style="background-color: aliceblue;">
  <img src="t_model.svg">
</div>

The biggest changes schematically is the splitting of the viscous loss impedance into halves and place each on either side of the path to the wall and switching to the admission model $G_t$ of the heat conduction loss and placed it in parallel with the yielding wall drop. So, now $H_w$ accounts for both heat conduction loss and yielding wall effects. The viscous loss $H_v$ has the same structure but halved values. Last, the fricative noise source $U_n$ is added at the junction for the completeness.

Setting up the pressure and flow conservations for this model:
$$\begin{align}
P_1 + P_e &= P_{v,1} + P_{v,2} + P_2,\\
U_1 + U_n &= U_w + U_2.
\end{align}$$

Express the viscous pressure losses and wall flow as functions of inputs and outputs:
$$\begin{align}
P_{v,1} &= H_v (U_1 + U_n) \notag\\
P_{v,2} &= H_v U_2 \notag\\
U_w &= H_w (P_{v,2}+P_2) \notag
\end{align}$$

Define the states of the $H_v$'s and $H_w$ as $s_{v,1}$, $s_{v,2}$, and $\mathbf{s}_w$, respectively, and also notate their state space coefficients in the same fashion. Then, the above equations can be written in the time domain as
$$\begin{align}
P_{v,1} &= c_v s_{v,1} + d_v (U_1+U_n)\\
P_{v,2} & = c_v s_{v,2} + d_v U_2\\
U_w &= \mathbf{c}_w \mathbf{s}_w + d_w (c_v s_{v,2} + d_v U_2 + P_2)
\end{align}$$

Substitute them into the conservation equations:
$$\begin{align}
P_1 + P_e &= c_v s_{v,1} + d_v (U_1+U_n) + c_v s_{v,2} + d_v U_2 + P_2,\\
U_1 + U_n &=  \mathbf{c}_w \mathbf{s}_w +  d_w (c_v s_{v,2} +  d_v U_2 + P_2) +  U_2\\
\end{align}$$

Neaten up a bit:
$$\begin{align}
P_2 + d_v U_2 &= P_1 - d_v U_1 + \beta,\\
(d_w d_v + 1)U_2 +  d_w P_2 &=  U_1 + \gamma,\\
\end{align}$$
where
$$\begin{align}
\beta &= P_e - c_v (s_{v,1} + s_{v,2}) - d_v U_n,\\
\gamma &= U_n - (d_w c_v s_{v,2} -  \mathbf{c}_w \mathbf{s}_w)\\
\end{align}$$

Convert to partial pressures:
$$\begin{align}
(F_2+B_2) + d_v Z_1^{-1}(F_2-B_2) &= (F_1+B_1) - d_v Z_1^{-1}(F_1-B_1) + \beta,\\
(d_w d_v + 1)Z_1^{-1}(F_2-B_2) +  d_w (F_2+B_2) &=  Z_1^{-1}(F_1-B_1) + \gamma,\\
\end{align}$$


Gather inputs and outputs:
$$\begin{align}
F_2 + d_v Z_1^{-1} F_2 - B_1 - d_v Z_1^{-1}B_1 &= F_1 - d_v Z_1^{-1}F_1 - B_2 + d_v Z_1^{-1}B_2+ \beta,\\
(d_w d_v + 1)Z_1^{-1} F_2 + d_w F_2 + Z_1^{-1}B_1 &=  Z_1^{-1} F_1 + (d_w d_v + 1)Z_1^{-1} B_2 - d_w B_2 + \gamma,\\
\end{align}$$


Convert to matrix-vector format and solve:
$$
\begin{bmatrix}
1 + d_v Z_1^{-1} & -1 -d_v Z_1^{-1}\\
(d_w d_v + 1)Z_1^{-1} + d_w & Z_1^{-1}
\end{bmatrix}
\begin{bmatrix}F_2\\B_1\end{bmatrix}
=
\begin{bmatrix}
1 - d_v Z_1^{-1} & -1 + d_v Z_1^{-1}\\
Z_1^{-1} & (d_w d_v + 1)Z_1^{-1} - d_w
\end{bmatrix}
\begin{bmatrix}F_1\\B_2\end{bmatrix}
+
\begin{bmatrix}\beta \\ \gamma\end{bmatrix}\tag{*}
$$


In [ ]:
dw, dv, Y1, Y2, D = symbols("d_w,d_v,Y_1,Y_2,D")
M = expand(Matrix([[1 + dv * Y1, -1 - dv * Y1], [(1 + dw * dv) * Y2 + dw, Y1]]))
Minv = simplify(M**-1)
cexpr = Y1 * dv + 1
dexpr = Y1 + Y2 * dv * dw + Y2 + dw
Minv = simplify(Minv * cexpr) * dexpr
display(Minv)
B = simplify(Minv * Matrix([[1, -1], [Y1, Y2 - dw]])).expand()
B = simplify((B - cexpr * dexpr * Matrix([[1, 0], [0, 1]])).expand())
display(B)
cv, cw = symbols(r"c_v,\mathbf{c}_w")
display(simplify(Minv * Matrix([[1, -dv, -cv, -cv, 0], [0, 1, 0, -dw * cv, -cw]])))

Matrix([
[                     Y_1, Y_1*d_v + 1],
[-Y_2*d_v*d_w - Y_2 - d_w, Y_1*d_v + 1]])

Matrix([
[-Y_1*Y_2*d_v**2*d_w - Y_1*Y_2*d_v - Y_1*d_v*d_w + Y_1 - Y_2*d_v*d_w - Y_2 - d_w,                        Y_1*Y_2*d_v - Y_1*d_v*d_w - Y_1 + Y_2 - d_w],
[                                     Y_1**2*d_v + Y_1 - Y_2*d_v*d_w - Y_2 - d_w, -Y_1**2*d_v - Y_1*Y_2*d_v**2*d_w - 2*Y_1*d_v*d_w - Y_1 + Y_2 - d_w]])

Matrix([
[                     Y_1,                                           1,                      -Y_1*c_v,         c_v*(-Y_1 - d_w*(Y_1*d_v + 1)), -\mathbf{c}_w*(Y_1*d_v + 1)],
[-Y_2*d_v*d_w - Y_2 - d_w, Y_1*d_v + d_v*(Y_2*d_v*d_w + Y_2 + d_w) + 1, c_v*(Y_2*d_v*d_w + Y_2 + d_w), c_v*(-Y_1*d_v*d_w + Y_2*d_v*d_w + Y_2), -\mathbf{c}_w*(Y_1*d_v + 1)]])

$$\begin{aligned}
\begin{bmatrix}F_2\\B_1\end{bmatrix}
=
\begin{bmatrix}F_1\\B_2\end{bmatrix}
&+
\frac{1}{D}
\begin{bmatrix}
-Z_1^{-1}Z_2^{-1}d_v^2d_w - Z_1^{-1}Z_2^{-1}d_v - Z_1^{-1}d_vd_w + Z_1^{-1} - Z_2^{-1}d_vd_w - Z_2^{-1} - d_w
& Z_1^{-1}Z_2^{-1}d_v - Z_1^{-1}d_vd_w -Z_1^{-1} + Z_2^{-1} - d_w\\
Z_1^{-2}d_v + Z_1^{-1} - Z_2^{-1}d_vd_w - Z_2^{-1} - d_w
& -Z_1^{-2}-Z_1^{-1}Z_2^{-1}d_v^2d_w - 2Z_1^{-1}d_vd_w - Z_1^{-1} + Z_2^{-1} - d_w
\end{bmatrix}
\begin{bmatrix}F_1\\B_2\end{bmatrix}\\
&\quad+
\frac{1}{D}
\begin{bmatrix}
Z_1^{-1} & Z_1^{-1} d_v + 1\\
Z_2^{-1} d_v d_w - Z_2^{-1} - d_w & Z_1^{-1} d_v + 1
\end{bmatrix}
\begin{bmatrix}\beta_n \\ \gamma_n\end{bmatrix}\\
=
\begin{bmatrix}F_1\\B_2\end{bmatrix}&+
\frac{(Z_1^{-1} - Z_2^{-1} - d_w)F_1 + (-Z_1^{-1} + Z_2^{-1} - d_w)B_2}{D}\\
&\quad+
\frac{Z_1^{-1} d_v}{D}
\begin{bmatrix}
- Z_2^{-1}(F_1 + B_2)\\
Z_1^{-1}(F_1-B_2)
\end{bmatrix}\\
&\quad-
\frac{d_v d_w}{D}
\begin{bmatrix}
Z_1^{-1}Z_2^{-1}d_v - Z_1^{-1} - Z_2^{-1} & Z_1^{-1}\\
Z_2^{-1} & Z_1^{-1}Z_2^{-1}d_v - 2Z_1^{-1}
\end{bmatrix}
\begin{bmatrix}F_1\\B_2\end{bmatrix}\\
&\quad+
\frac{1}{D}
\begin{bmatrix}
Z_1^{-1}\\
Z_2^{-1} d_v d_w - Z_2^{-1} - d_w
\end{bmatrix}
\beta
+
\frac{Z_1^{-1} d_v + 1}{D} \gamma
\end{aligned}$$
where
$$
D = (Z_1^{-1} d_v + 1)(Z_1^{-1} + Z_2^{-1} + d_w + Z_2^{-1} d_v d_w)

Now work on optimizing $\beta$ and $\gamma$
$$\begin{aligned}
\begin{bmatrix}\beta\\\gamma\end{bmatrix} &=
\begin{bmatrix}
P_e - c_v (s_{v,1} + s_{v,2}) - d_v U_n,\\
U_n - d_w c_v s_{v,2} + \mathbf{c}_w \mathbf{s}_w\\
\end{bmatrix}\\
&=
\begin{bmatrix}
1 & - d_v & - c_v & -c_v & \mathbf{0}\\
0 & 1 & 0 & - d_w c_v & -\mathbf{c}_w \\
\end{bmatrix}
\begin{bmatrix}
P_e \\ U_n \\ s_{v,1} \\ s_{v,2} \\ \mathbf{s}_w
\end{bmatrix}\\

\end{aligned}$$

Updating $H_w$ with the heat conduction loss admittance:

$$\begin{aligned}H_w &= \frac{C_ws}{L_wC_ws^2 + RC_ws + 1} + G_t\\
&= \frac{C_ws + G_t(L_wC_ws^2 + RC_ws + 1)}{L_wC_ws^2 + RC_ws + 1}\\
&= \frac{G_tL_wC_ws^2 + (1 + G_tR)C_ws + G_t}{L_wC_ws^2 + RC_ws + 1}\\
\end{aligned}$$

Use bilinear transformed $H_v$, and update both $s_{v,1}$ and $s_{v,2}$ together

# $\Pi$-model

While the T-model addresses all the interactions within a tube segment, it is unbalanced around the junction, leaving the effect of Tube 1 on the junction limited to the reflection. To balance the model, the half of each tube is now considered and the junction is centered. The number of states is now increased to 8.

<div style="background-color: aliceblue;">
  <img src="pi_model.svg">
</div>


Setting up the pressure and flow conservations for this model:
$$\begin{align}
P_1 -P_{v,11} - P_{v,12} + P_e &= P_{v,21} + P_{v,22} + P_2,\\
U_1 - U_{w,1} + U_n &= U_{w,2} + U_2.
\end{align}$$

$$
\begin{bmatrix}
1&0\\
0&1\\
\end{bmatrix}
\begin{bmatrix}P_1\\U_1\end{bmatrix}
+
\begin{bmatrix}
-1 & 0 & -1 & 1\\
0 & -1 & 0 & 0 & 1
\end{bmatrix}
\begin{bmatrix}P_{v,11}\\U_{w,1}\\P_{v,12}\\P_e\\U_n\end{bmatrix}
=
\begin{bmatrix}
1&0\\
0&1\\
\end{bmatrix}
\begin{bmatrix}P_2\\U_2\end{bmatrix}
+
\begin{bmatrix}
1 & 0 & 1\\
0 & 1 & 0
\end{bmatrix}
\begin{bmatrix}P_{v,22}\\U_{w,2}\\P_{v,21}\end{bmatrix}
$$


Express the viscous pressure losses and wall flow as functions of inputs and outputs:
$$\begin{align}
P_{v,11} &= H_{v,1} U_1 = d_{v,1} U_1 + c_{v,1} s_{v,11} \notag\\
U_{w,1} &= H_{w,1} (P_1 - P_{v,11}) = d_{w,1} P_1 - d_{w,1} d_{v,1} U_1 + d_{w,1} c_{v,1} s_{v,11} + \mathbf{c}_{w,1} \mathbf{s}_{w,1} \notag\\
P_{v,12} &= H_{v,1} (U_1-U_{w,1}) = (d_{v,1} - d_{v,1}^2 d_{w,1}) U_1 - d_{v,1} d_{w,1} P_1 + d_{v,1} d_{w,1} c_{v,1} s_{v,11} + c_{v,1}s_{v,12} + d_{v,1} \mathbf{c}_{w,1} \mathbf{s}_{w,1} \notag\\
P_{v,22} &= H_{v,2} U_2 = d_{v,2} U_2 + c_{v,2} s_{v,22} \notag\\
U_{w,2} &= H_{w,2} (P_{v,22}+P_2) = d_{w,2} d_{v,2} U_2 + d_{w,2} P_2 + d_{w,2} c_{v,2} s_{v,22} + \mathbf{c}_{w,2}\mathbf{s}_{w,2} \notag\\
P_{v,21} &= H_{v,2} (U_2 + U_{w,2}) = (d_{v,2} + d_{v,2}^2 d_{w,2}) U_2 + d_{v,2} d_{w,2} P_2 + d_{v,2} d_{w,2} c_{v,2} s_{v,22} + c_{v,2} s_{v,21} + d_{v,2} \mathbf{c}_{w,2}\mathbf{s}_{w,2}\notag\\
\end{align}$$

Express the viscous pressure losses and wall flow as functions of inputs and outputs:
$$\begin{align}
\begin{bmatrix}
P_{v,11}\\
U_{w,1}\\
P_{v,12}\\
P_e\\
U_n\\
\end{bmatrix}
&=
\begin{bmatrix}
0                 &                   d_{v,1}  \\
          d_{w,1} &         - d_{w,1} d_{v,1}  \\
- d_{v,1} d_{w,1} & d_{v,1} - d_{w,1} d_{v,1}^2\\
0 & 0\\
0 & 0\\
\end{bmatrix}
\begin{bmatrix}P_1\\U_1\\\end{bmatrix}
+
\begin{bmatrix}
0 & 0 &                 c_{v,1} & 0       & 0                       \\
0 & 0 &         d_{w,1} c_{v,1} & 0       &         \mathbf{c}_{w,1}\\
0 & 0 & d_{v,1} d_{w,1} c_{v,1} & c_{v,1} & d_{v,1} \mathbf{c}_{w,1}\\
1 & 0 & 0 & 0 & 0\\
0 & 1 & 0 & 0 & 0\\
\end{bmatrix}
\begin{bmatrix}
P_e\\
U_n\\
s_{v,11}\\
s_{v,12}\\
\mathbf{s}_{w,1}\\
\end{bmatrix}\\
\begin{bmatrix}
P_{v,22}\\
U_{w,2}\\
P_{v,21}\\
\end{bmatrix}
&=
\begin{bmatrix}
0               &                   d_{v,2}\\
        d_{w,2} &           d_{w,2} d_{v,2}\\
d_{v,2} d_{w,2} & d_{v,2} + d_{w,2} d_{v,2}^2\\
\end{bmatrix}
\begin{bmatrix}P_2\\U_2\\\end{bmatrix}
+
\begin{bmatrix}
0        &                 c_{v,2} & 0\\
0        &         d_{w,2} c_{v,2} &         \mathbf{c}_{w,2}\\
c_{v,2}  & d_{v,2} d_{w,2} c_{v,2} & d_{v,2} \mathbf{c}_{w,2}\\
\end{bmatrix}
\begin{bmatrix}
s_{v,21}\\
s_{v,22}\\
\mathbf{s}_{w,2}\\
\end{bmatrix}
\end{align}$$

$$
\begin{bmatrix}
1+d_{v,2} d_{w,2} & 2 d_{v,2} + d_{w,2} d_{v,2}^2\\
d_{w,2} & 1+d_{w,2} d_{v,2} \\
\end{bmatrix}
\begin{bmatrix}P_2\\U_2\\\end{bmatrix}
=
\begin{bmatrix}
1+d_{v,1} d_{w,1} & - 2d_{v,1} + d_{w,1} d_{v,1}^2\\
-d_{w,1} & 1+d_{w,1} d_{v,1} \\\end{bmatrix}
\begin{bmatrix}P_1\\U_1\\\end{bmatrix}
+\mathbf{\beta} +\mathbf{\gamma}
$$
where
$$\begin{align}
\mathbf{\beta} &= 
\begin{bmatrix}
-1 & 0 & c_{v,1}(1+d_{v,1} d_{w,1}) & c_{v,1} & d_{v,1} \mathbf{c}_{w,1}\\
0 & -1 & d_{w,1} c_{v,1} & 0 & \mathbf{c}_{w,1}
\end{bmatrix}
\begin{bmatrix}
P_e\\
U_n\\
s_{v,11}\\
s_{v,12}\\
\mathbf{s}_{w,1}\\
\end{bmatrix}\\
\mathbf{\gamma} &=
\begin{bmatrix}
c_{v,2} & c_{v,2} (1 + d_{v,2} d_{w,2}) & d_{v,2} \mathbf{c}_{w,2}\\
0 & d_{w,2} c_{v,2} & \mathbf{c}_{w,2}
\end{bmatrix}
\begin{bmatrix}
s_{v,21}\\
s_{v,22}\\
\mathbf{s}_{w,2}\\
\end{bmatrix}\\
\end{align}
$$



Convert to partial pressures:
$$
\begin{bmatrix}
1+d_{v,2} d_{w,2} & (2 d_{v,2} + d_{w,2} d_{v,2}^2)Z_2^{-1}\\
d_{w,2} & (1+d_{w,2} d_{v,2})Z_2^{-1}\\
\end{bmatrix}
\begin{bmatrix}F_2+B_2\\F_2-B_2\\\end{bmatrix}
=
\begin{bmatrix}
1+d_{v,1} d_{w,1} & (- 2d_{v,1} + d_{w,1} d_{v,1}^2)Z_1^{-1}\\
-d_{w,1} & 1+d_{w,1} d_{v,1}Z_1^{-1} \\\end{bmatrix}
\begin{bmatrix}F_1+B_1\\F_1-B_1\\\end{bmatrix}
+\mathbf{\beta} +\mathbf{\gamma}
$$

Gather $F_2$ and $B_1$ on the left hand side:
$$
\begin{bmatrix}
1+d_{v,2} d_{w,2} + (2 d_{v,2} + d_{w,2} d_{v,2}^2)Z_2^{-1} & -1-d_{v,1} d_{w,1} + (- 2d_{v,1} + d_{w,1} d_{v,1}^2)Z_1^{-1}\\
d_{w,2} + (1+d_{w,2} d_{v,2})Z_2^{-1} & d_{w,1} + 1+d_{w,1} d_{v,1}Z_1^{-1}\\
\end{bmatrix}
\begin{bmatrix}F_2\\B_1\end{bmatrix}
=
\begin{bmatrix}
1+d_{v,1} d_{w,1} + (- 2d_{v,1} + d_{w,1} d_{v,1}^2)Z_1^{-1} & -1-d_{v,2} d_{w,2} + (2 d_{v,2} + d_{w,2} d_{v,2}^2)Z_2^{-1}\\
-d_{w,1} + 1+d_{w,1} d_{v,1}Z_1^{-1} & -d_{w,2} + (1+d_{w,2} d_{v,2})Z_2^{-1}\\\end{bmatrix}
\begin{bmatrix}F_1\\B_2\end{bmatrix}
+\mathbf{\beta} +\mathbf{\gamma}
$$